# Regional Baseline — SARIMAX for All Categories x Flow Types x Hurricanes

**Purpose**: Fit SARIMAX baselines and export predicted vs actual for all combinations.
Once baselines are saved, metrics (largest drop, recovery, outflow increase) are computed
in a separate notebook without re-fitting models.

**Outputs**: `results/regional_level/{hurricane}/baseline_{flow}_{category}.csv`
Each CSV contains: date, y_true, y_pred, ci_lower, ci_upper

**Plots**: Predicted vs actual for every category x flow (saved to figures/)

In [1]:
import pandas as pd
import numpy as np
import h5py
import os
import sys
import datetime as dt
from importlib import reload

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

folder_path = './../../hurricane_oct/'
sys.path.append(folder_path)
sys.path.append(os.path.join(folder_path, 'mobility_function'))
from mobility_function import analysis as ma
ma = reload(ma)

%run ./recovery_function_v2.py

OUTPUT_ROOT = '../results/regional_level'
print('Setup complete.')

Setup complete.


In [2]:
# ── Date helpers ──
def mondays_str(year, start_month=7, end_month=10):
    start = dt.date(year, start_month, 28)
    end = dt.date(year, end_month, 31)
    days_ahead = (0 - start.weekday()) % 7
    cur = start + dt.timedelta(days=days_ahead)
    out = []
    while cur <= end:
        out.append(cur.strftime('%Y%m%d'))
        cur += dt.timedelta(days=7)
    return out

mondays_2023 = mondays_str(2023)
mondays_2024 = mondays_str(2024)
all_mondays = mondays_2023 + mondays_2024

dates_2023 = pd.date_range(start='2023-07-31', periods=len(mondays_2023)*7, freq='D')
dates_2024 = pd.date_range(start='2024-07-29', periods=len(mondays_2024)*7, freq='D')
dates_all = dates_2023.union(dates_2024)

GROUPS = {
    'Travel':              [1, 2, 3, 4, 5, 12],
    'Work & Professional': [9],
    'Health':              [14],
    'Education':           [13],
    'Retail & Leisure':    [11, 15],
    'Urban Government':    [6, 7, 8, 16],
    'Utilities':           [0, 10],
}
group_names = list(GROUPS.keys())

# SARIMAX specification
ARIMA_ORDER = (1, 0, 0)
SEASONAL_ORDER = (0, 0, 0, 0)

print(f'Mondays: {len(all_mondays)}, Total days: {len(dates_all)}')
print(f'Categories: {len(group_names)}')
print(f'ARIMA: {ARIMA_ORDER}, Seasonal: {SEASONAL_ORDER}')

Mondays: 28, Total days: 196
Categories: 7
ARIMA: (1, 0, 0), Seasonal: (0, 0, 0, 0)


In [3]:
# ── Hurricane configs ──
HURRICANE_CONFIGS = {
    'milton': {
        'landing_date': pd.Timestamp('2024-10-09'),
        'cutoff_mile': 50,
    },
    'helene': {
        'landing_date': pd.Timestamp('2024-09-26'),
        'cutoff_mile': 50,
    },
}

FLOW_TYPES = ['within', 'inflow', 'outflow']

In [5]:
# ── Load mobility data for both hurricanes ──
def load_regional_flows(hrc_name, cutoff_mile):
    """Load H5 files and decompose into within/outflow/inflow at regional level."""
    with open(f'../results/{hrc_name}/counties_geoid_cut_{cutoff_mile}.txt', 'r') as f:
        county_list = [int(line.strip()) for line in f]
    
    geo_idx = pd.read_csv('geoid_idx_names.csv')
    geo_idx['GEOID'] = geo_idx['GEOID'].astype(int)
    selected_idx = geo_idx[geo_idx['GEOID'].isin(county_list)].county_idx.values
    print(f'  {hrc_name}: {len(selected_idx)} counties')
    
    # Load and decompose
    W_all, O_all, I_all = [], [], []
    for date_str in all_mondays:
        M = ma.h5py_to_4d_array(folder_path + f'data/mobility/M_raw_{date_str}.h5')
        M_w, M_o, M_i = ma.region_mobility(M, selected_idx)
        # Sum over counties: (days, 17, n_counties) -> (days, 17)
        W_all.append(M_w.sum(axis=2))
        O_all.append(M_o.sum(axis=2))
        I_all.append(M_i.sum(axis=2))
        print(f'    {date_str} done')
    
    W_ts = np.concatenate(W_all, axis=0)  # (total_days, 17)
    O_ts = np.concatenate(O_all, axis=0)
    I_ts = np.concatenate(I_all, axis=0)
    
    # Merge 17 categories -> 7 groups
    n_days = W_ts.shape[0]
    n_groups = len(group_names)
    
    def merge_cats(raw_ts):
        merged = np.zeros((n_days, n_groups))
        for g, name in enumerate(group_names):
            idx = GROUPS[name]
            merged[:, g] = raw_ts[:, idx].sum(axis=1)
        return merged
    
    return {
        'within': merge_cats(W_ts),
        'outflow': merge_cats(O_ts),
        'inflow': merge_cats(I_ts),
    }

# Load both hurricanes
all_flow_data = {}
for hrc_name, cfg in HURRICANE_CONFIGS.items():
    print(f'\nLoading {hrc_name}...')
    all_flow_data[hrc_name] = load_regional_flows(hrc_name, cfg['cutoff_mile'])
    print(f'  Shape: {all_flow_data[hrc_name]["within"].shape}')


Loading milton...
  milton: 21 counties
    20230731 done
    20230807 done
    20230814 done
    20230821 done
    20230828 done
    20230904 done
    20230911 done
    20230918 done
    20230925 done
    20231002 done
    20231009 done
    20231016 done
    20231023 done
    20231030 done
    20240729 done
    20240805 done
    20240812 done
    20240819 done
    20240826 done
    20240902 done
    20240909 done
    20240916 done
    20240923 done
    20240930 done
    20241007 done
    20241014 done
    20241021 done
    20241028 done
  Shape: (196, 7)

Loading helene...
  helene: 271 counties
    20230731 done
    20230807 done
    20230814 done
    20230821 done
    20230828 done
    20230904 done
    20230911 done
    20230918 done
    20230925 done
    20231002 done
    20231009 done
    20231016 done
    20231023 done
    20231030 done
    20240729 done
    20240805 done
    20240812 done
    20240819 done
    20240826 done
    20240902 done
    20240909 done
    20240916 done

## Fit SARIMAX Baselines and Export

For each hurricane x flow type x category:
1. Fit SARIMAX on pre-hurricane training data
2. Generate predictions with 95% CI
3. Save CSV: date, y_true, y_pred, ci_lower, ci_upper
4. Plot predicted vs actual

In [6]:
# ── Fit and export all baselines ──
for hrc_name, hrc_cfg in HURRICANE_CONFIGS.items():
    landing_date = hrc_cfg['landing_date']
    train_end = (landing_date - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    forecast_start = (landing_date - pd.Timedelta(days=6)).strftime('%Y-%m-%d')
    forecast_end = '2024-10-31'
    
    out_dir = f'{OUTPUT_ROOT}/{hrc_name}'
    fig_dir = f'{out_dir}/figures'
    os.makedirs(fig_dir, exist_ok=True)
    
    flow_data = all_flow_data[hrc_name]
    
    print(f"\n{'='*70}")
    print(f'HURRICANE: {hrc_name.upper()} (landing: {landing_date.date()})')
    print(f"{'='*70}")
    
    for ft_name in FLOW_TYPES:
        ts_array = flow_data[ft_name]  # (days, 7)
        
        for g, cat_name in enumerate(group_names):
            safe_cat = cat_name.replace(' ', '_').replace('&', 'and')
            csv_path = f'{out_dir}/baseline_{ft_name}_{safe_cat}.csv'
            fig_path = f'{fig_dir}/baseline_{ft_name}_{safe_cat}.png'
            
            flow_y = ts_array[:, g]
            
            try:
                y_log, y, X = prepare_time_series_with_exog(flow_y, dates_all)
                res, y_train, X_train = fit_arimax_model(
                    y_log, X,
                    order=ARIMA_ORDER, seasonal_order=SEASONAL_ORDER,
                    train_2024_end=train_end)
                df_rec, forecast_idx = get_predictions_and_ci(
                    res, X, y,
                    forecast_start=forecast_start, forecast_end=forecast_end)
                
                # Save baseline CSV
                df_rec.to_csv(csv_path)
                
                # Plot: predicted vs actual
                fig, ax = plt.subplots(figsize=(14, 5))
                ax.plot(df_rec.index, df_rec['y_true'], 'k-', lw=1.5, label='Observed')
                ax.plot(df_rec.index, df_rec['y_pred'], 'r-', lw=1.5, label='Predicted (baseline)')
                ax.fill_between(df_rec.index, df_rec['ci_lower'], df_rec['ci_upper'],
                               color='red', alpha=0.15, label='95% CI')
                ax.axvline(landing_date, color='blue', ls='--', lw=2, label='Landing date')
                ax.set_title(f'{hrc_name.capitalize()} — {cat_name} — {ft_name}\n'
                            f'Predicted vs Actual', fontweight='bold', fontsize=13)
                ax.set_xlabel('Date')
                ax.set_ylabel('Mobility (flow)')
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.savefig(fig_path, dpi=150, bbox_inches='tight')
                plt.close()
                
                print(f'  [{ft_name}] {cat_name}: saved ({len(df_rec)} days)')
                
            except Exception as e:
                print(f'  [{ft_name}] {cat_name}: FAILED — {e}')
    
    # List all saved baselines
    n_files = len([f for f in os.listdir(out_dir) if f.startswith('baseline_') and f.endswith('.csv')])
    print(f'\n  Total baselines saved: {n_files} CSVs + {n_files} figures')


HURRICANE: MILTON (landing: 2024-10-09)
  [within] Travel: saved (29 days)
  [within] Work & Professional: saved (29 days)
  [within] Health: saved (29 days)
  [within] Education: saved (29 days)
  [within] Retail & Leisure: saved (29 days)
  [within] Urban Government: saved (29 days)
  [within] Utilities: saved (29 days)
  [inflow] Travel: saved (29 days)
  [inflow] Work & Professional: saved (29 days)
  [inflow] Health: saved (29 days)
  [inflow] Education: saved (29 days)
  [inflow] Retail & Leisure: saved (29 days)
  [inflow] Urban Government: saved (29 days)
  [inflow] Utilities: saved (29 days)
  [outflow] Travel: saved (29 days)
  [outflow] Work & Professional: saved (29 days)
  [outflow] Health: saved (29 days)
  [outflow] Education: saved (29 days)
  [outflow] Retail & Leisure: saved (29 days)
  [outflow] Urban Government: saved (29 days)
  [outflow] Utilities: saved (29 days)

  Total baselines saved: 21 CSVs + 21 figures

HURRICANE: HELENE (landing: 2024-09-26)
  [within] T

/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  [within] Work & Professional: saved (42 days)
  [within] Health: saved (42 days)
  [within] Education: saved (42 days)


/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  [within] Retail & Leisure: saved (42 days)
  [within] Urban Government: saved (42 days)
  [within] Utilities: saved (42 days)
  [inflow] Travel: saved (42 days)
  [inflow] Work & Professional: saved (42 days)
  [inflow] Health: saved (42 days)
  [inflow] Education: saved (42 days)
  [inflow] Retail & Leisure: saved (42 days)
  [inflow] Urban Government: saved (42 days)
  [inflow] Utilities: saved (42 days)
  [outflow] Travel: saved (42 days)
  [outflow] Work & Professional: saved (42 days)
  [outflow] Health: saved (42 days)
  [outflow] Education: saved (42 days)
  [outflow] Retail & Leisure: saved (42 days)
  [outflow] Urban Government: saved (42 days)
  [outflow] Utilities: saved (42 days)

  Total baselines saved: 21 CSVs + 21 figures


In [7]:
# ── Verification: show all saved baselines ──
for hrc_name in HURRICANE_CONFIGS:
    out_dir = f'{OUTPUT_ROOT}/{hrc_name}'
    files = sorted([f for f in os.listdir(out_dir) if f.startswith('baseline_') and f.endswith('.csv')])
    print(f'\n{hrc_name}: {len(files)} baseline CSVs')
    for f in files:
        df = pd.read_csv(f'{out_dir}/{f}', index_col=0, parse_dates=True)
        print(f'  {f}: {len(df)} days, pred range [{df["y_pred"].min():.0f}, {df["y_pred"].max():.0f}]')


milton: 21 baseline CSVs
  baseline_inflow_Education.csv: 29 days, pred range [66231, 112363]
  baseline_inflow_Health.csv: 29 days, pred range [366220, 564728]
  baseline_inflow_Retail_and_Leisure.csv: 29 days, pred range [2222167, 3086331]
  baseline_inflow_Travel.csv: 29 days, pred range [303347, 362569]
  baseline_inflow_Urban_Government.csv: 29 days, pred range [38558, 56461]
  baseline_inflow_Utilities.csv: 29 days, pred range [1430, 2316]
  baseline_inflow_Work_and_Professional.csv: 29 days, pred range [2473030, 3127082]
  baseline_outflow_Education.csv: 29 days, pred range [88370, 169931]
  baseline_outflow_Health.csv: 29 days, pred range [495605, 822636]
  baseline_outflow_Retail_and_Leisure.csv: 29 days, pred range [2022356, 2823656]
  baseline_outflow_Travel.csv: 29 days, pred range [255720, 325861]
  baseline_outflow_Urban_Government.csv: 29 days, pred range [49180, 75447]
  baseline_outflow_Utilities.csv: 29 days, pred range [1893, 2514]
  baseline_outflow_Work_and_Profes